## Data Cleaning 

In [67]:
import pandas as pd

years = list(range(2015, 2026))

# import CAD data
cad_list = []
for year in years:
    df = pd.read_csv(f"Eugene_CAD_data_noloc/EugeneCAD{year}noloc.csv", low_memory=False)
    cad_list.append(df)

cad = pd.concat(cad_list, ignore_index=True)
cad.head()

,yr,agency,service,inci_id,calltime,case_id,callsource,nature,closecode,closed_as,secs_to_disp,secs_to_arrv,secs_to_close,disp,arrv,priority,primeunit,units_dispd,units_arrived,month
0,2015,EPD,LAW,15000001,2015-01-01 00:00:00.000,NaN,SELF,PERSON STOP,ASST,ASSISTED,0.0,0.0,217,1,1,6,_5E48,1,1,NaN
1,2015,EPD,LAW,15000002,2015-01-01 00:00:44.000,NaN,SELF,FIGHT,RSLV,RESOLVED,0.0,0.0,2114,1,1,P,_3F65,4,2,NaN
2,2015,EPD,LAW,15000003,2015-01-01 00:01:05.000,NaN,PHONE,CHECK WELFARE,ASST,ASSISTED,708.0,1094.0,1538,1,1,5,_3J79,1,1,NaN
3,2015,EPD,LAW,15000007,2015-01-01 00:03:16.000,NaN,W911,SHOTS FIRED,PCHK,PATROL CHECK,277.0,576.0,891,1,1,3,_5E48,2,2,NaN
4,2015,EPD,LAW,15000010,2015-01-01 00:03:34.000,NaN,SELF,ILLEGAL FIREWORKS,ADVI,ADVISED,0.0,0.0,150,1,1,5,_5K97,1,1,NaN


In [68]:
# clean cad
cad_clean = cad[["calltime", "nature", "priority", "agency", "primeunit"]].copy()

# fix whitespace
cad_clean["agency"] = cad_clean["agency"].astype(str).str.strip()
cad_clean["primeunit"] = cad_clean["primeunit"].astype(str).str.strip()

# convert time
cad_clean["call_time"] = pd.to_datetime(cad_clean["calltime"], errors="coerce")

# rename
cad_clean = cad_clean.rename(columns={"nature": "call_type"})

# 1. PRIMARY RESPONDER (what actually handled the call)
cad_clean["response_prime"] = cad_clean["primeunit"].apply(lambda x: "CAHOOTS" if x == "_CAHOT" else "EPD")

# 2. AGENCY INVOLVEMENT (who was assigned / involved)
cad_clean["response_agency"] = cad_clean["agency"].apply(lambda x: "CAHOOTS" if x == "CAHE" else "EPD")

# keep final columns 
cad_clean = cad_clean[["call_time", "call_type", "priority", "response_prime", "response_agency"]]

cad_clean.head()

,call_time,call_type,priority,response_prime,response_agency
0,2015-01-01 00:00:00,PERSON STOP,6,EPD,EPD
1,2015-01-01 00:00:44,FIGHT,P,EPD,EPD
2,2015-01-01 00:01:05,CHECK WELFARE,5,EPD,EPD
3,2015-01-01 00:03:16,SHOTS FIRED,3,EPD,EPD
4,2015-01-01 00:03:34,ILLEGAL FIREWORKS,5,EPD,EPD


In [69]:
# import MCSLC data
mcslc = pd.read_excel("MCSLC.xlsx")
mcslc.head()

,ID,End Point of Dispatch,City,Dispatch Request Date & Time,Dispatch Date & Time,Arrival on Scene Date & Time,Engagement with Client Date & Time,MCIT Departure Date & Time,Reason for Dispatch #1,Reason for Dispatch #2,Reason for Dispatch #3,Reason for Dispatch #4,Reason for Dispatch #5,Disposition,Minutes: Request → Dispatch,Minutes: Dispatch → Arrival,Minutes: Arrival → Engagement,Minutes: Arrival → Departure
0,12236,Cancelled,Eugene,2025-05-12 13:29:00,2025-05-12 13:34:00,2025-05-12 00:00:00,2025-05-12 00:00:00,2025-05-12 14:03:00,Harm/risk of harm to self/others/property,Disorganized behavior,Agitation or disruptive behavior,NaN,NaN,Arrest,5.0,NaN,0.0,NaN
1,12574,Cancelled,Unknown,2025-12-05 17:41:00,2025-12-05 17:47:00,2025-12-05 18:06:00,NaT,2025-12-05 18:12:00,Agitation or disruptive behavior,NaN,NaN,NaN,NaN,Arrest,6.0,19.0,NaN,NaN
2,12888,Cancelled,Eugene,2025-07-25 21:10:00,2025-07-25 21:14:00,2025-07-25 21:28:00,2025-07-25 21:30:00,2025-07-25 21:36:00,Agitation or disruptive behavior,NaN,NaN,NaN,NaN,Arrest,4.0,14.0,2.0,8.0
3,10504,Engaged Client,Eugene,2025-03-29 22:29:00,2025-03-29 22:43:00,2025-03-29 23:21:00,2025-03-29 23:28:00,2025-03-29 23:58:00,Harm/risk of harm to self/others/property,Disorganized behavior,Agitation or disruptive behavior,NaN,NaN,Arrest,14.0,38.0,7.0,37.0
4,12220,Engaged Client,Eugene,2025-04-26 18:52:00,2025-04-26 18:54:00,2025-04-26 19:05:00,2025-04-26 19:07:00,2025-04-26 19:32:00,Disorganized behavior,Agitation or disruptive behavior,NaN,NaN,NaN,Arrest,2.0,11.0,2.0,27.0


In [70]:
# clean MCSLC and filter to only Eugene
mcslc_clean = mcslc[mcslc["City"] == "Eugene"].copy()

# create call_time
mcslc_clean["call_time"] = pd.to_datetime(mcslc_clean["Dispatch Request Date & Time"], errors="coerce")

# rename call type
mcslc_clean["call_type"] = mcslc_clean["Reason for Dispatch #1"]

# match both response columns
mcslc_clean["response_prime"] = "MCSLC"
mcslc_clean["response_agency"] = "MCSLC"

# priority variable
mcslc_clean["priority"] = None

# keep final columns
mcslc_clean = mcslc_clean[["call_time", "call_type", "priority", "response_prime", "response_agency"]]

mcslc_clean.head()

,call_time,call_type,priority,response_prime,response_agency
0,2025-05-12 13:29:00,Harm/risk of harm to self/others/property,None,MCSLC,MCSLC
2,2025-07-25 21:10:00,Agitation or disruptive behavior,None,MCSLC,MCSLC
3,2025-03-29 22:29:00,Harm/risk of harm to self/others/property,None,MCSLC,MCSLC
4,2025-04-26 18:52:00,Disorganized behavior,None,MCSLC,MCSLC
5,2025-05-12 20:48:00,Disorganized behavior,None,MCSLC,MCSLC


In [71]:
# combine datasets
combined = pd.concat([cad_clean, mcslc_clean], ignore_index=True)

# add time variables
combined["year"] = combined["call_time"].dt.year.astype("Int64")
combined["month"] = combined["call_time"].dt.month.astype("Int64")

In [72]:
# create pre/post CAHOOTS shutdown variable
import numpy as np

shutdown_date = pd.to_datetime("2025-04-05")

combined["period"] = np.where(combined["call_time"] < shutdown_date, "pre", "post")

In [73]:
combined.head()

,call_time,call_type,priority,response_prime,response_agency,year,month,period
0,2015-01-01 00:00:00,PERSON STOP,6,EPD,EPD,2015,1,pre
1,2015-01-01 00:00:44,FIGHT,P,EPD,EPD,2015,1,pre
2,2015-01-01 00:01:05,CHECK WELFARE,5,EPD,EPD,2015,1,pre
3,2015-01-01 00:03:16,SHOTS FIRED,3,EPD,EPD,2015,1,pre
4,2015-01-01 00:03:34,ILLEGAL FIREWORKS,5,EPD,EPD,2015,1,pre


In [74]:
# save to csv
combined.to_csv("combined_calls_data.csv", index=False)